# Imports

In [1]:
import os
import sys
import operator

# To define constants
from typing_extensions import Literal

# Langchain LLM model integration
from langchain.chat_models import init_chat_model

# To support older python versions
from typing_extensions import Optional, Annotated, List, Sequence

# Different Message types for defining chat model ip and op
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, get_buffer_string

# Langgraph Entities to build and design control flow
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command

# Persistence Layer - Checkpoints 
from langgraph.checkpoint.memory import InMemorySaver

# Chat History Management to append messages
from langgraph.graph.message import add_messages

# For runtime validation on state and output schemas
from pydantic import BaseModel, Field

# Format Markdowns
from rich.markdown import Markdown

# Custom Imports
sys.path.append(os.path.abspath("../src"))
from prompts import clarify_with_user_instructions, transform_messages_into_research_topic_prompt
from utils import format_markdown_messages

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

C:\SK7 Officials\langgraph-deep-research\langchain_env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

# Defining State and Schemas

## State Definitions

In [2]:
class AgentInputState(MessagesState):
    """Input state with only messages from user input."""
    pass

class AgentMasterState(MessagesState):
    """
    Main state for the full multi-agent research system.
    Extends MessagesState with additional fields for research coordination.
    """

    # Research brief generated from user conversation history
    research_brief: Optional[str]
    # Messages exchanged with the supervisor agent for coordination
    supervisor_messages: Annotated[Sequence[BaseMessage], add_messages]
    # Raw unprocessed research notes collected during the research phase
    raw_notes: Annotated[list[str], operator.add] = []
    # Processed and structured notes ready for report generation
    notes: Annotated[list[str], operator.add] = []
    # Final formatted research report
    final_report: str

## Structured Output Schemas

In [3]:
class ClarifyWithUserSchema(BaseModel):
    """Schema for user clarification decision and questions."""
    
    need_clarification: bool = Field(
        description="Whether the user needs to be asked a clarifying question.",
    )
    question: str = Field(
        description="A question to ask the user to clarify the report scope",
    )
    verification: str = Field(
        description="Verify message that we will start research after the user has provided the necessary information.",
    )

class ResearchQuestionSchema(BaseModel):
    """Schema for structured research brief generation."""
    
    research_brief: str = Field(
        description="A research question that will be used to guide the research.",
    )

# Scope Research Workflow

*User Clarification and Research Brief Generation*

This module implements the scoping phase of the research workflow, where we:
1. Assess if the user's request needs clarification
2. Generate a detailed research brief from the conversation

The workflow uses structured output to make deterministic decisions about
whether sufficient context exists to proceed with research.
"""

## Setting Up LLM

In [4]:
llm = init_chat_model(
    "openai:gpt-4.1",
    temperature=0, # Factual response
    max_tokens=1000, # OpenRouter free token constraints
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1"
)

## Defining Nodes

In [5]:
def clarify_with_user(state: AgentMasterState) -> Command[Literal["write_research_brief", "__end__"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.
    """
    # Set up structured output from the LLM
    structured_output_model = llm.with_structured_output(ClarifyWithUserSchema)

    # Invoke the model with clarification instructions and user conversations
    # get_buffer_string => list of messages to a string
    response = structured_output_model.invoke([HumanMessage(content=clarify_with_user_instructions.format(messages=get_buffer_string(
                                                                                                                    messages=state.get("messages", []))))])
    
    # Route based on clarification need
    # If clarification is needed, ask ; else generate research brief
    if response.need_clarification:
        return Command(
            goto=END, 
            update={"messages": [AIMessage(content=response.question)]} # Follow up question is added to AI Messge history
        )
    else:
        return Command(
            goto="write_research_brief", 
            update={"messages": [AIMessage(content=response.verification)]} # Verification acknowledgement is added to AI Message history
        )

def write_research_brief(state: AgentMasterState):
    """
    Transform the conversation history into a comprehensive research brief.
    """
    # Set up structured output from the LLM
    structured_output_model = llm.with_structured_output(ResearchQuestionSchema)
    
    # Generate research brief from conversation history)

    response = structured_output_model.invoke([HumanMessage(content=transform_messages_into_research_topic_prompt.format(
                                                                                            messages=get_buffer_string(messages=state.get("messages", []))))])
    
    # Update state with generated research brief and pass it to the supervisor
    return {
        "research_brief": response.research_brief,
        "supervisor_messages": [HumanMessage(content=f"{response.research_brief}.")]
    }


## Defining Workflow

In [6]:
# Build the scoping workflow
deep_researcher_builder = StateGraph(AgentMasterState, input_schema=AgentInputState)

# Add workflow nodes
deep_researcher_builder.add_node("clarify_with_user", clarify_with_user)
deep_researcher_builder.add_node("write_research_brief", write_research_brief)

# Add workflow edges
deep_researcher_builder.add_edge(START, "clarify_with_user") # END if further claification needed, else goto write_research_brief
deep_researcher_builder.add_edge("write_research_brief", END)

# Compile the workflow
scope_research = deep_researcher_builder.compile(checkpointer=InMemorySaver())

## Sample Conversation

In [7]:
# Thread to store the checkpoints
thread = {"configurable": {"thread_id": "1"}}

result = scope_research.invoke({"messages": [HumanMessage(content="I want to research the all time best tamil movies.")]}, config=thread)

In [8]:
format_markdown_messages(result['messages'])

### 🧑 Human
I want to research the all time best tamil movies.

### 🤖 AI
Could you clarify what criteria you would like to use for determining the 'all time best' Tamil movies? For example, are you interested in critical acclaim, box office success, audience ratings, awards won, or a combination of these factors? Also, do you want movies from all eras or only a specific time period?



In [9]:
result = scope_research.invoke({"messages": [HumanMessage(content="Let's examine 2000s films only and consider combination of all factors.")]}, config=thread)

In [10]:
format_markdown_messages(result['messages'])

### 🧑 Human
I want to research the all time best tamil movies.

### 🤖 AI
Could you clarify what criteria you would like to use for determining the 'all time best' Tamil movies? For example, are you interested in critical acclaim, box office success, audience ratings, awards won, or a combination of these factors? Also, do you want movies from all eras or only a specific time period?

### 🧑 Human
Let's examine 2000s films only and consider combination of all factors.

### 🤖 AI
Thank you for clarifying your criteria. You are interested in researching the all-time best Tamil movies from the 2000s, considering a combination of critical acclaim, box office success, audience ratings, and awards. I have sufficient information and will now begin the research process based on these parameters.



In [11]:
Markdown(result["research_brief"])

I want to identify and analyze the all-time best Tamil movies released during the 2000s (from 2000 to 2009), using 
a combination of the following factors: critical acclaim (including reviews from reputable critics and film        
festivals), box office success (measured by gross earnings and commercial performance), audience ratings (from     
platforms such as IMDb and Rotten Tomatoes, if available), and awards won (including national, state, and          
international recognitions). I have not specified any particular genre, director, or actor preferences, so all     
types of films should be considered. Unless otherwise noted, consider all available sources, prioritizing official 
box office records, reputable film review aggregators, and official award listings. If there are additional        
relevant factors for determining 'best' that are commonly used in film studies but not specified here, please note 
them as open considerations rather than assumed requirements.